# [TABLE_NAME] ETL

## Purpose
[Describe what this table does]

## Input → Output
* **Source:** `big_data.[source_schema].[source_table]`
* **Target:** `big_data.[target_schema].[target_table]`
* **Primary Key:** `[column_name]`

## Transformations
1. [Step 1 description]
2. [Step 2 description]
3. [Add more as needed]

## Data Quality
* **Technical:** NOT NULL (PK), UNIQUE (PK), critical columns validation
* **Business:** [Add domain-specific rules if needed, or set `validation_business = True`]

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, 
    DoubleType, 
    DecimalType, 
    BooleanType, 
    StringType,
    TimestampType
)

In [0]:
# Schema configuration
source_schema = "big_data.[source_schema]"
target_schema = "big_data.[target_schema]"

# Source Tables (full paths)
source_table1 = f"{source_schema}.[source_table_name1]"
source_table2 = f"{source_schema}.[source_table_name2]"
# e.g.
# oders = f"{source_schema}.orders"
# order_products_prior = f"{source_schema}.order_products_prior"

source_tables = [
    source_table1, 
    source_table2,
]

# Target Table (full path)
target_table = f"{target_schema}.[target_table_name]"
# e.g.
# target_table = f"{target_schema}.new_orders"

# Primary Key columns for validation
primary_key_columns = ["[column_name]"]  # List format: ["id"] or ["order_id", "product_id"]

# Critical columns (NOT NULL required)
critical_columns = []  # e.g., ["order_date", "quantity", "status"]

# Print configuration
print("Configuration:")
print(f"  Source_tables: {source_tables}")
print(f"  Target: {target_table}")
print(f"  Primary Key: {primary_key_columns}")

### TRANSFORMATION

In [0]:
# put all cells that transform data starting here

In [0]:
# e.g.
# print("Step 1: Loading sources and casting types...")

# # Load Bronze table and apply type casting
# df_source1 = spark.table(source_table1) \
#     .withColumn("[col1]", F.col("[col1]").cast(IntegerType())) \
#     .withColumn("[col2]", F.col("[col2]").cast(DoubleType())) \
#     .withColumn("[col3]", F.col("[col3]").cast(DecimalType(10, 2))) \
#     .withColumn("[col4]", F.trim(F.col("[col4]"))) \
#     .filter(F.col(primary_key_columns[0]).isNotNull())
#     # Add more type casts and basic cleaning as needed

# print(f"  Loaded: {df_source1.count():,} rows after type casting and NULL filtering")



In [0]:
# put as many transformation cells as needed until here

In [0]:
# rename df to df_result for validations

df_result = df_source1

### DATA QUALITY

In [0]:
# Execute technical validations using UTILS orchestrator
validation_technical, total_rows = technical_validations(
    df=df_result,
    primary_key_columns=primary_key_columns,
    critical_columns=critical_columns,
    range_checks=[]  # Add if needed: [("hour", 0, 23), ("day", 1, 31)]
)

In [0]:
# Business validations (optional)
# If no business rules needed, just leave this as True
validation_business = True

# Example: Check expected values
# expected_statuses = ['active', 'completed', 'cancelled']
# invalid = df_result.filter(~F.col('status').isin(expected_statuses)).count()
# if invalid > 0:
#     print(f"⚠️ Found {invalid} rows with invalid status")
#     validation_business = False

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, target_table)
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")